# DFA Lab — train a tiny character RNN

**Language:** binary strings ending in `0`. **Goal:** compare learned predictions with an exact DFA.

Run every cell in order in Google Colab using a CPU runtime. No uploaded data or API key is required. This notebook creates its own working folder and training scripts. Training uses 193 learned parameters, 15 epochs, and fixed seeds.

This is a **sequence classifier**, not a generative language model. Neural prediction is empirical; the DFA is exact. Perfect test accuracy is not a proof for all strings.


In [ ]:
%pip -q install numpy==2.2.6 matplotlib


## 1. Create the project files
These are the same readable modules included with the Flask project. Re-running the notebook replaces these files and training artifacts.


In [ ]:
from pathlib import Path
import os
project = Path('/content/dfa-lab') if Path('/content').exists() else Path.cwd() / 'dfa-colab-run'
project.mkdir(parents=True, exist_ok=True)
os.chdir(project)
print('Working folder:', project)


### Exact DFA and validation

In [ ]:
%%writefile dfa.py
"""Exact recognizer for L = {w in {0,1}* : w ends in 0}."""
MAX_LENGTH = 256


def validate(text):
    if not isinstance(text, str):
        raise ValueError('string must be a JSON string.')
    if len(text) > MAX_LENGTH:
        raise ValueError(f'Use at most {MAX_LENGTH} characters.')
    if any(c not in '01' for c in text):
        raise ValueError('Only 0 and 1 are allowed; spaces are not removed.')


def run_dfa(text):
    validate(text)
    state = 'q0'
    trace = [{'step': 0, 'symbol': None, 'state': state}]
    for i, symbol in enumerate(text, 1):
        state = 'q1' if symbol == '0' else 'q0'
        trace.append({'step': i, 'symbol': symbol, 'state': state})
    return {'accepted': state == 'q1', 'final_state': state, 'trace': trace}


### RNN forward pass, gradients, and model export

In [ ]:
%%writefile model.py
"""Tiny character RNN. All parameters are learned, including recurrent weights.

h_t = tanh(x_t W + h_(t-1) U + b); P(accept) = sigmoid(h_final v + c).
Padding freezes hidden states, so it never becomes a character.
"""
import numpy as np


class RNN:
    def __init__(self, hidden=12, seed=42):
        rng = np.random.default_rng(seed)
        self.p = dict(W=rng.normal(0, .25, (2, hidden)),
                      U=rng.normal(0, .15, (hidden, hidden)),
                      b=np.zeros(hidden), v=rng.normal(0, .25, hidden), c=np.zeros(1))

    def forward(self, strings, cache=False):
        n = len(strings)
        h = np.zeros((n, len(self.p['b'])))
        history = []
        for t in range(max(map(len, strings), default=0)):
            active = np.array([t < len(s) for s in strings])[:, None]
            x = np.zeros((n, 2))
            for i, s in enumerate(strings):
                if t < len(s):
                    x[i, int(s[t])] = 1
            candidate = np.tanh(x @ self.p['W'] + h @ self.p['U'] + self.p['b'])
            if cache:
                history.append((x, h, candidate, active))
            h = np.where(active, candidate, h)
        z = h @ self.p['v'] + self.p['c'][0]
        prob = 1 / (1 + np.exp(-np.clip(z, -40, 40)))
        return (prob, h, history) if cache else prob

    def gradients(self, strings, labels):
        prob, h, history = self.forward(strings, cache=True)
        dz = (prob - labels) / len(labels)
        grad = {k: np.zeros_like(v) for k, v in self.p.items()}
        grad['v'] = h.T @ dz
        grad['c'][0] = dz.sum()
        dh = dz[:, None] * self.p['v']
        for x, previous, candidate, active in reversed(history):
            da = dh * active * (1 - candidate ** 2)
            grad['W'] += x.T @ da
            grad['U'] += previous.T @ da
            grad['b'] += da.sum(axis=0)
            dh = da @ self.p['U'].T + dh * (~active)
        loss = -np.mean(labels * np.log(prob + 1e-9) + (1-labels) * np.log(1-prob + 1e-9))
        return float(loss), grad

    def save(self, path):
        np.savez_compressed(path, **self.p)

    @classmethod
    def load(cls, path):
        model = cls()
        with np.load(path, allow_pickle=False) as data:
            model.p = {k: data[k].copy() for k in model.p}
        return model


### Dataset generation, Adam training, and held-out evaluation

In [ ]:
%%writefile train.py
"""Generate unique labeled examples, train with Adam, evaluate, and export."""
import argparse
import csv
import itertools
import json
from pathlib import Path
import numpy as np
from dfa import run_dfa
from model import RNN

ROOT = Path(__file__).resolve().parent


def generate(seed=42):
    rng = np.random.default_rng(seed)
    seen = set()
    while len(seen) < 6000:
        length = int(rng.integers(1, 25))
        seen.add(''.join(map(str, rng.integers(0, 2, length))))
    strings = sorted(seen)  # Stable even across Python processes.
    rng.shuffle(strings)
    splits = {'train': [''] + strings[:4200], 'validation': strings[4200:5100],
              'test': strings[5100:]}
    ood = set()
    while len(ood) < 1000:
        ood.add(''.join(map(str, rng.integers(0, 2, int(rng.integers(25, 129))))))
    splits['long_test'] = sorted(ood)
    (ROOT / 'data').mkdir(exist_ok=True)
    for split, values in splits.items():
        with (ROOT / 'data' / f'{split}.csv').open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['string', 'label', 'length'])
            writer.writerows((s, int(run_dfa(s)['accepted']), len(s)) for s in values)
    return splits


def evaluate(model, strings):
    y = np.array([run_dfa(s)['accepted'] for s in strings], dtype=int)
    p = np.concatenate([model.forward(strings[i:i+128]) for i in range(0, len(strings), 128)])
    pred = (p >= .5).astype(int)
    cm = [[int(((y == a) & (pred == b)).sum()) for b in (0, 1)] for a in (0, 1)]
    return {'count': len(strings), 'accuracy': float((pred == y).mean()),
            'confusion_matrix': cm, 'matrix_order': 'rows=true, columns=predicted; reject, accept',
            'mismatches': [{'string': s, 'exact': int(a), 'predicted': int(b)}
                           for s, a, b in zip(strings, y, pred) if a != b][:20]}


def main(epochs=15):
    splits = generate()
    model = RNN()
    rng = np.random.default_rng(43)
    m = {k: np.zeros_like(v) for k, v in model.p.items()}
    v = {k: np.zeros_like(value) for k, value in model.p.items()}
    step, best_loss, history = 0, float('inf'), []
    (ROOT / 'artifacts').mkdir(exist_ok=True)
    for epoch in range(1, epochs+1):
        order = rng.permutation(len(splits['train']))
        losses = []
        for start in range(0, len(order), 128):
            batch = [splits['train'][i] for i in order[start:start+128]]
            labels = np.array([run_dfa(s)['accepted'] for s in batch], dtype=float)
            loss, grad = model.gradients(batch, labels)
            losses.append(loss)
            step += 1
            norm = np.sqrt(sum(np.sum(g*g) for g in grad.values()))
            for k in model.p:
                g = grad[k] * min(1., 1. / (norm + 1e-9))
                m[k] = .9*m[k] + .1*g
                v[k] = .999*v[k] + .001*g*g
                model.p[k] -= .01 * (m[k]/(1-.9**step)) / (np.sqrt(v[k]/(1-.999**step)) + 1e-8)
        val = splits['validation']
        p = model.forward(val)
        y = np.array([run_dfa(s)['accepted'] for s in val])
        val_loss = float(-np.mean(y*np.log(p+1e-9)+(1-y)*np.log(1-p+1e-9)))
        history.append({'epoch': epoch, 'train_loss': float(np.mean(losses)), 'validation_loss': val_loss})
        if val_loss < best_loss:
            best_loss = val_loss
            model.save(ROOT / 'artifacts' / 'model.npz')
        print(f'Epoch {epoch:02d}: train loss {np.mean(losses):.5f}, validation loss {val_loss:.5f}', flush=True)
    model = RNN.load(ROOT / 'artifacts' / 'model.npz')
    exhaustive = [''.join(s) for length in range(11) for s in itertools.product('01', repeat=length)]
    report = {'seed': 42, 'architecture': 'character RNN, 12 tanh hidden units, sigmoid output',
              'parameters': sum(p.size for p in model.p.values()), 'threshold': .5,
              'training_lengths': [0, 24], 'epochs': epochs, 'history': history,
              'test': evaluate(model, splits['test']), 'long_test': evaluate(model, splits['long_test']),
              'exhaustive_0_to_10': evaluate(model, exhaustive),
              'note': 'Exhaustive set overlaps training. Test and long_test are held out. Accuracy is empirical, not proof.'}
    (ROOT / 'artifacts' / 'metrics.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps({k: report[k] for k in ('test', 'long_test', 'exhaustive_0_to_10')}, indent=2))


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--epochs', type=int, default=15)
    args = parser.parse_args()
    if args.epochs < 1:
        parser.error('--epochs must be positive')
    main(args.epochs)


## 2. Generate, train, evaluate, and export

The training script creates 4,201 training examples (including ε), 900 validation examples, 900 test examples, and 1,000 longer examples. No string appears in more than one split. Labels come from the exact DFA. The model sees one-hot characters and learns with binary cross-entropy and backpropagation through time. Padding freezes the hidden state. Validation loss selects the exported checkpoint; test labels are never used for optimization.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "train.py", "--epochs", "15"], check=True)


## 3. Inspect actual evaluation and the learning curve
Confusion matrix rows are true labels and columns are predictions, both ordered reject then accept. The exhaustive test overlaps training; only test and long_test are held out.


In [ ]:
import json
import matplotlib.pyplot as plt
metrics = json.loads(Path('artifacts/metrics.json').read_text())
for split in ('test', 'long_test', 'exhaustive_0_to_10'):
    result = metrics[split]
    print(f"{split}: {result['accuracy']:.2%} of {result['count']} strings")
    print('Confusion matrix:', result['confusion_matrix'])
    print('First mismatches:', result['mismatches'])
history = metrics['history']
plt.figure(figsize=(8, 4))
plt.plot([r['epoch'] for r in history], [r['train_loss'] for r in history], label='Training')
plt.plot([r['epoch'] for r in history], [r['validation_loss'] for r in history], label='Validation')
plt.xlabel('Epoch'); plt.ylabel('Binary cross-entropy'); plt.title('Learning to recognize strings ending in 0')
plt.legend(); plt.grid(alpha=.2); plt.tight_layout(); plt.show()


## 4. Inspect data and verify no split leakage
Empty strings must remain empty; they are not missing values.


In [ ]:
import csv
import itertools
splits = {}
for name in ('train', 'validation', 'test', 'long_test'):
    with open(f'data/{name}.csv', newline='') as file:
        rows = list(csv.DictReader(file))
    splits[name] = {r['string'] for r in rows}
    assert len(splits[name]) == len(rows)
    print(name, len(rows), 'accepted:', sum(int(r['label']) for r in rows))
for a, b in itertools.combinations(splits, 2):
    assert not splits[a].intersection(splits[b])
assert '' in splits['train']
print('Unique examples, disjoint splits, and empty-string handling verified.')


## 5. Compare predictions and verify export
Try changing the examples. The probability is P(accept), not guaranteed confidence. Longer strings test generalization.


In [ ]:
import numpy as np
from model import RNN
from dfa import run_dfa
model = RNN.load('artifacts/model.npz')
examples = ['', '0', '1', '1010', '1011', '10'*50]
before = model.forward(examples)
model.save('artifacts/roundtrip.npz')
after = RNN.load('artifacts/roundtrip.npz').forward(examples)
np.testing.assert_array_equal(before, after)
Path('artifacts/roundtrip.npz').unlink()
for text, probability in zip(examples, before):
    print(repr(text), 'P(accept)=', round(float(probability), 5),
          'neural=', bool(probability >= .5), 'DFA=', run_dfa(text)['accepted'])
print('Export/reload predictions are identical.')


## 6. Download the trained model

Download the ZIP, extract its `artifacts` folder, and copy it into the local Flask project, replacing that folder. Restart Flask. Both the weights and metrics must come from the same run. Colab is used for training; the website runs locally.


In [ ]:
import zipfile
with zipfile.ZipFile('dfa-trained-artifacts.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ('model.npz', 'metrics.json'):
        archive.write(Path('artifacts')/name, f'artifacts/{name}')
try:
    from google.colab import files
except ImportError:
    print('Saved:', project / 'dfa-trained-artifacts.zip')
else:
    files.download('dfa-trained-artifacts.zip')


## What to report

- Explain the two states and transition table, including why ε is rejected.
- Describe the split sizes, encoding, architecture, optimization, and selection by validation loss.
- Report test and long-test accuracy and confusion matrices from your run.
- Explain why the DFA is authoritative and why finite neural testing cannot establish universal correctness.
- This easy language depends only on the last character: the neural model demonstrates learning, not a better validator.
